# 1. Out-of-Distribution (OOD) Experiments

Two experiments test whether DynaGuide can steer the robot using visual guidance under conditions outside the training distribution.

**Experiment 1 — OOD Color Change (`NutAssemblyRedSquare`):**  
The square nut is replaced by a geometrically identical red nut. The base policy degrades because it was never trained on red objects. DynaGuide provides new visual guidance examples collected from the red-nut environment to test whether visual guidance alone can recover performance.

**Experiment 2 — Two-Nut Steering (`NutAssemblyDualSquare`):**  
Both a gold and a red square nut are placed on the table in the same spawn region. The base policy has no preference between them. By swapping the positive/negative guidance files, DynaGuide steers the robot to preferentially pick up either nut.

> **Fair comparison:** For a valid comparison between the base policy and DynaGuide, both must start from the **exact same robot and nut positions** in every episode. Each experiment section includes the matched baseline command alongside the DynaGuide command — both read from the same `initial_states.json`.

**Experiment 3 — Insertion Pose Steering Experiment:**
Compare the success rates & and number of insertions from a specific direction when guiding only into a choosen singular direction.</br>
Optional: Add specific directions as negative guidance samples to influence insertion pose even more.</br>
The goal is to evaluated DynaGuides potantial fine-grained influence not only on the pure performance of the Diffusion Policy but also on the specific direction from which the nut is inserted during a rollout.

## 1.1 Environment Overview

| Environment | Nut(s) | Purpose |
|---|---|---|
| `NutAssemblySquare` | 1 × gold square nut | In-distribution reference |
| `NutAssemblyRedSquare` | 1 × red square nut | OOD color-change experiment |
| `NutAssemblyDualSquare` | 1 × gold + 1 × red (shared zone) | Two-nut steering evaluation |
| `NutAssemblyGuidanceGold` | gold at training pos + red random | Guidance collection: clean gold pickups |
| `NutAssemblyGuidanceRed` | red at training pos + gold random | Guidance collection: clean red pickups |

The two **Guidance** environments exploit the fact that the base policy navigates by position, not colour. By placing the target nut exactly where the policy expects it (the training spawn zone: x ≈ −0.11 m, y ∈ [0.11, 0.225] m), the policy picks it up reliably regardless of colour. The distractor nut is placed on the opposite side of the table (y ∈ [−0.25, −0.05] m) so it never draws the policy away. This gives clean, label-guaranteed pickup examples for both colours without needing a successful dual-square policy.

**`NutAssemblyDualSquare` spawn zone:** Both nuts share the same region as the training zone. This removes position as a confound for the evaluation — only DynaGuide's visual signal can drive nut selection during the actual steering experiment.

## 1.2 Verify the New Environments

Run the cell below to confirm both OOD environments load correctly and spawn the expected nuts:

In [ ]:
import sys
sys.path.insert(0, 'robosuite')
import numpy as np
import robosuite

# Evaluation environments
for env_name in ['NutAssemblyRedSquare', 'NutAssemblyDualSquare']:
    env = robosuite.make(
        env_name, robots='Panda',
        has_renderer=False, has_offscreen_renderer=False,
        use_camera_obs=False, use_object_obs=True,
    )
    obs = env.reset()
    print(f'{env_name} nuts: {[n.name for n in env.nuts]}')
    env.close()

# Guidance-collection environments — verify spawn zones
print()
for env_name, target in [('NutAssemblyGuidanceGold', 'SquareNut'),
                           ('NutAssemblyGuidanceRed',  'RedSquareNut')]:
    env = robosuite.make(
        env_name, robots='Panda',
        has_renderer=False, has_offscreen_renderer=False,
        use_camera_obs=False, use_object_obs=True,
    )
    target_ys, distractor_ys, dists = [], [], []
    for _ in range(5):
        obs = env.reset()
        sq  = obs['SquareNut_pos']
        red = obs['RedSquareNut_pos']
        target_pos     = sq  if target == 'SquareNut' else red
        distractor_pos = red if target == 'SquareNut' else sq
        target_ys.append(target_pos[1])
        distractor_ys.append(distractor_pos[1])
        dists.append(float(np.linalg.norm(sq[:2] - red[:2])))
    env.close()
    print(f'{env_name}:')
    print(f'  target ({target}) y:     min={min(target_ys):.3f}  max={max(target_ys):.3f}  (expected 0.11–0.225)')
    print(f'  distractor y:            min={min(distractor_ys):.3f}  max={max(distractor_ys):.3f}  (expected -0.25–-0.05)')
    print(f'  separation:              min={min(dists):.3f}  max={max(dists):.3f}  (all > 0.22: {all(d>0.22 for d in dists)})')

NutAssemblyRedSquare nuts: ['RedSquareNut']
NutAssemblyDualSquare nuts: ['SquareNut', 'RedSquareNut']

NutAssemblyGuidanceGold:
  target (SquareNut) y:     min=0.124  max=0.219  (expected 0.11–0.225)
  distractor y:            min=-0.202  max=-0.127  (expected -0.25–-0.05)
  separation:              min=0.321  max=0.439  (all > 0.22: True)
NutAssemblyGuidanceRed:
  target (RedSquareNut) y:     min=0.111  max=0.170  (expected 0.11–0.225)
  distractor y:            min=-0.241  max=-0.100  (expected -0.25–-0.05)
  separation:              min=0.232  max=0.414  (all > 0.22: True)


## 1.3 Matched Evaluation — How Initial States Are Shared

`dyna_scripts/run_trained_agent_in_order.py` is the single script for all policy runs — both data collection and baseline evaluation. The difference is whether `--initial_states_json` is supplied:

| Script | `--initial_states_json` | Reset mechanism | Use for |
|---|---|---|---|
| `run_trained_agent_in_order.py` | not provided | `env.reset()` — random | Initial data collection |
| `run_trained_agent_in_order.py` | provided | `env.reset_to(state)` | Baseline evaluation (matched) |
| `run_dynaguide.py` | via `reset_poses` in exp config | `env.reset_to(state)` | DynaGuide evaluation (matched) |

Baseline evaluation and DynaGuide both read from the **same `initial_states.json`**, so robot and nut positions are identical in every episode across all conditions.

> State vector length differs between environments — `NutAssemblyRedSquare` has fewer objects in the XML than `NutAssemblyDualSquare`. Each environment needs its own `initial_states.json`.

### Creating an `initial_states.json`

The pattern is the same for every environment. These steps are listed inline per experiment below.

**Step 1 — Collect rollouts with random resets** (successes not required — only diverse starting positions needed):
```bash
mkdir dataset/NutAssemblySquare
python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --env NutAssemblySquare \
    --n_rollouts 100 --horizon 200 \
    --dataset_path dataset/<env>/rollouts.hdf5 \
    --dataset_obs --seed 0
```

**Step 2 — Extract the first MuJoCo state from each rollout:**
```bash
python helper_scripts/create_reset_poses.py \
    --input dataset/<env>/rollouts.hdf5 \
    --output dataset/<env>/initial_states.json
```

The same `rollouts.hdf5` is also the source for guidance examples (positive and negative HDF5 files).

---
## 1.4 Experiment 1 — OOD Color Change (Red Nut)

The base policy was trained only on gold nuts. This experiment measures the success-rate drop under visual OOD and tests whether DynaGuide's visual guidance — using examples from the red-nut environment — can recover it.

### 1.4.1 Prepare Data

**Step 1 — Collect rollouts in `NutAssemblyRedSquare`:**

In [ ]:
!mkdir -p dataset/ood_red_nut
!python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --env NutAssemblyRedSquare \
    --n_rollouts 100 --horizon 200 \
    --dataset_path dataset/ood_red_nut/rollouts.hdf5 \
    --dataset_obs --seed 0

**Step 2 — Extract initial states** (used by both baseline and DynaGuide):

> `NutAssemblyRedSquare` has a different XML model than `NutAssemblySquare` — always extract initial states from rollouts run *inside* the target environment.

In [ ]:
!python helper_scripts/create_reset_poses.py \
    --input dataset/ood_red_nut/rollouts.hdf5 \
    --output dataset/ood_red_nut/initial_states.json

**Step 3 — Add proprio to rollouts:**

In [ ]:
!python helper_scripts/add_proprio_to_dataset.py \
    --input dataset/ood_red_nut/rollouts.hdf5

**Step 4 — Extract guidance examples:**

In [ ]:
!python helper_scripts/create_guidance_dataset.py \
    --input dataset/ood_red_nut/rollouts.hdf5 \
    --output dataset/ood_red_nut/red_nut_pos_examples.hdf5 \
    --camera agentview_image --snapshot_mode success --behavior success

In [ ]:
!python helper_scripts/create_guidance_dataset.py \
    --input dataset/ood_red_nut/rollouts.hdf5 \
    --output dataset/ood_red_nut/red_nut_neg_examples.hdf5 \
    --camera agentview_image --snapshot_mode peg_center_fail --behavior failure

**Step 5 — Inspect and filter guidance samples:**

In [ ]:
!mkdir -p eval/ood_red_nut/pos_images eval/ood_red_nut/neg_images
!python helper_scripts/show_guidance_images.py \
    --dataset dataset/ood_red_nut/red_nut_pos_examples.hdf5 \
    --output eval/ood_red_nut/pos_images \
    --camera agentview_image

In [ ]:
!python helper_scripts/show_guidance_images.py \
    --dataset dataset/ood_red_nut/red_nut_neg_examples.hdf5 \
    --output eval/ood_red_nut/neg_images \
    --camera agentview_image

In [ ]:
# Remove unwanted samples by 0-based index (adjust --exclude list after inspecting images)
!python helper_scripts/filter_guidance_dataset.py \
    --input dataset/ood_red_nut/red_nut_pos_examples.hdf5 \
    --exclude 1 4

In [ ]:
!python helper_scripts/filter_guidance_dataset.py \
    --input dataset/ood_red_nut/red_nut_neg_examples.hdf5 \
    --exclude 0 2

### 1.4.2 Baseline — Base Policy Only (no guidance)

Same initial states as DynaGuide (via `--initial_states_json`) for a direct success-rate comparison:

In [ ]:
!python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --env NutAssemblyRedSquare \
    --n_rollouts 100 --horizon 200 \
    --initial_states_json dataset/ood_red_nut/initial_states.json \
    --video_path evaluation/ood/baseline_red_nut.mp4 \
    --json_path evaluation/ood/baseline_red_nut_stats.json \
    --seed 0

### 1.4.3 DynaGuide — Visual Guidance (red nut)

In [ ]:
!python dyna_scripts/run_dynaguide.py \
    --ood_mode red_nut \
    --output_folder evaluation/ood/ \
    --dp_model models/dp_ep500.pth \
    --dynamics_model models/model_epoch_999_augmented_200.pth \
    --exp_setup_config configs/experiments/ood_configs/ood_red_nut.json \
    --camera_names agentview_image \
    --n_rollouts 100 --horizon 200 --seed 0 \
    --render_denoising

### 1.4.4 DSRL (red nut)

This guide expects that you have already worked trough [DSRL](DSRL.ipynb). and therefore are familiar with handeling DSRL in this environment.

After having obtained a suitable noise vector following [DSRL](DSRL.ipynb). use the following command to evaluute DSRLs performance when being confronted with a red nut insteadt of the gold one:

```bash
python dsrl_scripts/dsrl_test.py \
    --dp_ckpt dataset/YOUR_PREPARED_DP_CKPT.pth \
    --noise_actor_ckpt dataset/YOUR_NOISE_ACTOR.pth \
    --n_episodes 100 \
    --max_steps 200 \
    --video_path ./RESULT_FOLDER \
    --env NutAssemblyRedSquare
```

---
## 1.5 Experiment 2 — Two-Nut Steering (Dual Square)

Both a gold and a red square nut are placed in the same spawn region. The base policy has no preference. By swapping pos/neg guidance files, DynaGuide steers the robot toward either nut.



### 1.5.1 Prepare Data

#### A. Initial states — for evaluation only

Collect random-reset rollouts in `NutAssemblyDualSquare` to bootstrap the initial state JSON:

In [ ]:
!mkdir -p dataset/dual_square
!python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --env NutAssemblyDualSquare \
    --n_rollouts 100 --horizon 200 \
    --dataset_path dataset/dual_square/rollouts.hdf5 \
    --dataset_obs --seed 0

In [ ]:
!python helper_scripts/create_reset_poses.py \
    --input dataset/dual_square/rollouts.hdf5 \
    --output dataset/dual_square/initial_states.json

#### B. Guidance examples — one environment per nut colour

Collect clean, label-guaranteed single-colour pickup trajectories using the Guidance environments:

In [ ]:
!python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --env NutAssemblyGuidanceGold \
    --n_rollouts 100 --horizon 200 \
    --dataset_path dataset/dual_square/gold_pickup_rollouts.hdf5 --dataset_obs \
    --json_path evaluation/guidance_collection/gold_pickup_stats.json --seed 0

In [ ]:
!python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --env NutAssemblyGuidanceRed \
    --n_rollouts 100 --horizon 200 \
    --dataset_path dataset/dual_square/red_pickup_rollouts.hdf5 --dataset_obs \
    --json_path evaluation/guidance_collection/red_pickup_stats.json --seed 0

In [ ]:
!python helper_scripts/add_proprio_to_dataset.py \
    --input dataset/dual_square/gold_pickup_rollouts.hdf5

In [ ]:
!python helper_scripts/add_proprio_to_dataset.py \
    --input dataset/dual_square/red_pickup_rollouts.hdf5

Extract guidance frames from all successful rollouts:

In [ ]:
!python helper_scripts/create_guidance_dataset.py \
    --input dataset/dual_square/gold_pickup_rollouts.hdf5 \
    --output dataset/dual_square/gold_nut_on_peg.hdf5 \
    --camera agentview_image --snapshot_mode success --behavior success

In [ ]:
!python helper_scripts/create_guidance_dataset.py \
    --input dataset/dual_square/red_pickup_rollouts.hdf5 \
    --output dataset/dual_square/red_nut_on_peg.hdf5 \
    --camera agentview_image --snapshot_mode success --behavior success

Inspect guidance images and filter outliers:

In [ ]:
!mkdir -p evaluation/dual_square/gold_images evaluation/dual_square/red_images
!python helper_scripts/show_guidance_images.py \
    --dataset dataset/dual_square/gold_nut_on_peg.hdf5 \
    --output evaluation/dual_square/gold_images \
    --camera agentview_image

In [ ]:
!python helper_scripts/show_guidance_images.py \
    --dataset dataset/dual_square/red_nut_on_peg.hdf5 \
    --output evaluation/dual_square/red_images \
    --camera agentview_image

In [ ]:
# Adjust --exclude indices after inspecting images above
!python helper_scripts/filter_guidance_dataset.py \
    --input dataset/dual_square/gold_nut_on_peg.hdf5 \
    --exclude 3

In [ ]:
!python helper_scripts/filter_guidance_dataset.py \
    --input dataset/dual_square/red_nut_on_peg.hdf5 \
    --exclude 5

### 1.5.2 Baseline — No Preference

Measures the natural gold/red nut-selection split. Uses the same initial states as both DynaGuide conditions:

In [ ]:
!python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --env NutAssemblyDualSquare \
    --n_rollouts 100 --horizon 200 \
    --initial_states_json dataset/dual_square/initial_states.json \
    --video_path evaluation/ood/dual_baseline.mp4 \
    --json_path evaluation/ood/dual_baseline_stats.json \
    --seed 0

### 1.5.3 DynaGuide — Steer Toward Red Nut

In [ ]:
!python dyna_scripts/run_dynaguide.py \
    --ood_mode dual_red \
    --output_folder evaluation/ood/ \
    --dp_model models/dp_ep500.pth \
    --dynamics_model models/model_epoch_999_augmented_200.pth \
    --exp_setup_config configs/experiments/ood_configs/dual_pick_red.json \
    --camera_names agentview_image \
    --n_rollouts 100 --horizon 200 --seed 0 \
    --render_denoising

### 1.5.4 DynaGuide — Steer Toward Gold Nut

Same command as above — only `--ood_mode dual_gold` changes, swapping the pos/neg guidance files:

In [ ]:
!python dyna_scripts/run_dynaguide.py \
    --ood_mode dual_gold \
    --output_folder evaluation/ood/ \
    --dp_model models/dp_ep500.pth \
    --dynamics_model models/model_epoch_999_augmented_200.pth \
    --exp_setup_config configs/experiments/ood_configs/dual_pick_gold.json \
    --camera_names agentview_image \
    --n_rollouts 100 --horizon 200 --seed 0 \
    --render_denoising

---
## 1.6 Automatic Result Tracking: Insertion Angles & Orientation Clusters

Both `run_dynaguide.py` and `run_trained_agent_in_order.py` automatically track the nut's insertion angle at the end of every successful rollout — for **all** NutAssembly environments.

### What is tracked

**Insertion yaw (`assembly_yaw_deg`):** The yaw angle (degrees, world frame) of the nut body when it is on the peg, read from MuJoCo's `body_xquat`. Tells you from which direction the robot approached the peg.

**Orientation cluster:** Each yaw is mapped to one of four robot-relative quadrants:

| Cluster | Yaw range | Meaning |
|---|---|---|
| Left (45–135°) | 45° – 135° | Robot inserted from the left side |
| Back (135–225°) | 135° – 225° | Robot inserted facing toward itself |
| Right (225–315°) | 225° – 315° | Robot inserted from the right side |
| Front (315–45°) | 315° – 45° | Robot inserted from behind |

### Console output after every run

```
=== Insertion Angle Summary ===
  All yaw angles (deg): [87.3, 91.2, 85.1, ...]
  Mean: 88.9°  Std: 2.8°  Min: 84.1°  Max: 94.3°

  Orientation Clusters:
    Left (45-135°)    :  12/12 (100%)  ############
    Back (135-225°)   :   0/12 (  0%)
    Right (225-315°)  :   0/12 (  0%)
    Front (315-45°)   :   0/12 (  0%)
```

For `NutAssemblyDualSquare`, an additional per-nut breakdown is printed:

```
=== Per-Nut Assembly Summary ===
  No assembly:    3/20 rollouts (15%)
  RedSquareNut:  14/20 rollouts (70%)  mean yaw=87.3° (std 12.1°)
      Left (45-135°)        :  10/14 (71%)  ##########
      Back (135-225°)       :   4/14 (29%)  ####
      Right (225-315°)      :   0/14 ( 0%)
      Front (315-45°)       :   0/14 ( 0%)
  SquareNut:      3/20 rollouts (15%)  mean yaw=91.0° (std 8.4°)
      Left (45-135°)        :   3/ 3 (100%)  ###
```

### Saved output files

`run_trained_agent_in_order.py` automatically saves three files next to the primary output (derived from `--json_path`, `--video_path`, or `--dataset_path`, in that priority):

| File | Contents |
|---|---|
| `<name>_stats.json` (your `--json_path`) | Aggregated rollout statistics including yaw summary |
| `<name>_log.txt` | Full console output captured during the run |
| `<name>_args.json` | Exact command line and parsed arguments used |

Example: `--json_path evaluation/ood/baseline_red_nut_stats.json` produces `baseline_red_nut_stats_log.txt` and `baseline_red_nut_stats_args.json` in the same folder.

### Video overlay

On the final (success) frame, yaw angle, cluster, and nut name (DualSquare) are rendered in yellow text for quick visual verification without re-running.

### Implementation

- `NutAssembly.assembled_nut_yaw()` — base-class method inherited by all NutAssembly subclasses; reads `sim.data.body_xquat` of the first assembled nut
- `NutAssemblyDualSquare.which_nut_assembled()` — additionally reports which nut is on the peg
- Both scripts unwrap robomimic wrappers via `_unwrap_env()` and check for these methods with `hasattr()` — silently skipped on non-NutAssembly environments

## 1.7 Expected Results Summary

| Experiment | Environment | Condition | Expected outcome |
|---|---|---|---|
| In-distribution reference | `NutAssemblySquare` | Base policy | Current best success rate |
| OOD baseline | `NutAssemblyRedSquare` | Base policy | Drop vs. in-distribution |
| OOD + visual guidance | `NutAssemblyRedSquare` | Visual (red data) | Recovery toward in-distribution |
| Dual baseline | `NutAssemblyDualSquare` | Base policy | ~50/50 gold/red split |
| Dual → red | `NutAssemblyDualSquare` | Visual (red pos, gold neg) | Red nut preference ↑ |
| Dual → gold | `NutAssemblyDualSquare` | Visual (gold pos, red neg) | Gold nut preference ↑ |

Both experiments isolate visual guidance as the only variable: same policy checkpoint, same initial states, only the guidance condition changes. The red-nut experiment tests **whether** visual guidance works under OOD; the dual-nut experiment tests **how precisely** it can steer between two visually distinct options.

---
# 2. Tight-Fit Insertion Experiment (`NutAssemblyTightSquare`)

This experiment tests whether DynaGuide can recover insertion success when the inner hole of the square nut is narrowed, making precise alignment with the peg mandatory.

**Geometry:**
- Peg cross-section: 32 × 32 mm (half-size 0.016 m)
- Original nut inner hole: ~45.5 × 45.5 mm (half-size ~0.02275 m) → ~6.75 mm clearance per side
- Default tight hole: 36 × 36 mm (`--hole_x 0.018 --hole_y 0.018`) → ~2 mm clearance per side
- Clearance can be tuned: stricter values (e.g. 0.017) increase difficulty further

**Fair comparison:** Both baseline and DynaGuide use the **same `initial_states.json`** collected from `NutAssemblySquare`, so robot and nut starting positions are identical. Only the nut geometry changes between conditions.

| Parameter | Default | Tight | Very Tight |
|---|---|---|---|
| `hole_x` / `hole_y` (half-extent, m) | 0.02275 | 0.018 | 0.017 |
| Clearance per side (mm) | 6.75 | 2.0 | 1.0 |

## 2.1 Verify the Environment

Confirm the tight nut loads and the hole geometry is as expected:

In [1]:
import sys
sys.path.insert(0, 'robosuite')
import robosuite

for hole in [0.02275, 0.018, 0.017]:
    env = robosuite.make(
        'NutAssemblyTightSquare', robots='Panda',
        has_renderer=False, has_offscreen_renderer=False,
        use_camera_obs=False, use_object_obs=True,
        hole_x=hole, hole_y=hole,
    )
    obs = env.reset()
    clearance_mm = (hole - 0.016) * 1000
    print(f'hole_x=hole_y={hole:.4f} m  →  clearance={clearance_mm:.1f} mm  |  nut: {[n.name for n in env.nuts]}')
    env.close()

[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /home/maxi/ss26_g2_smlr/robosuite/robosuite/scripts/setup_macros.py (macros.py:55)


hole_x=hole_y=0.0227 m  →  clearance=6.7 mm  |  nut: ['SquareNut']
hole_x=hole_y=0.0180 m  →  clearance=2.0 mm  |  nut: ['SquareNut']
hole_x=hole_y=0.0170 m  →  clearance=1.0 mm  |  nut: ['SquareNut']


## 2.2 Prepare Initial States

Reuse the existing `NutAssemblySquare` initial states so starting positions are identical to the in-distribution baseline. No new data collection is required.

If the initial states file does not yet exist, create it now (uses the `NutAssemblySquare` rollouts from Section 7.3):

In [ ]:
!python helper_scripts/create_reset_poses.py \
    --input dataset/square_splits/square_valid.hdf5 \
    --output dataset/square_splits/square_initial_states.json

## 2.3 Baseline — Base Policy with Tight Nut (no guidance)

`--hole_x` and `--hole_y` automatically switch the environment to `NutAssemblyTightSquare` with the given inner hole half-extents. Same initial states as the standard evaluation for a direct comparison:

In [ ]:
!python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --n_rollouts 100 --horizon 200 \
    --initial_states_json dataset/square_splits/square_initial_states.json \
    --dataset_path dataset/tight_square/baseline_rollouts.hdf5 \
    --dataset_obs \
    --hole_x 0.018 --hole_y 0.018 \
    --json_path eval/tight_square/baseline_tight_stats.json \
    --seed 0 \
    --video_path eval/tight_square/baseline_tight.mp4

## 2.4 Prepare Guidance Data for the Tight-Fit Environment

The guidance images need to show the nut inserted on the peg in the tight-fit environment. Collect rollouts from the tight-fit environment — successful ones can serve directly as positive guidance:

In [ ]:
!mkdir -p dataset/tight_square
!python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --n_rollouts 100 --horizon 200 \
    --dataset_path dataset/tight_square/rollouts.hdf5 \
    --dataset_obs \
    --hole_x 0.018 --hole_y 0.018 \
    --seed 0

In [ ]:
!python helper_scripts/add_proprio_to_dataset.py \
    --input dataset/tight_square/rollouts.hdf5

Extract positive guidance frames (successful insertions in the tight environment):

In [ ]:
!python helper_scripts/create_guidance_dataset.py \
    --input dataset/tight_square/rollouts.hdf5 \
    --output dataset/tight_square/pos_examples.hdf5 \
    --camera agentview_image \
    --snapshot_mode success \
    --behavior success

Inspect and filter the guidance samples:

In [ ]:
!mkdir -p evaluation/tight_square/pos_images
!python helper_scripts/show_guidance_images.py \
    --dataset dataset/tight_square/pos_examples.hdf5 \
    --output evaluation/tight_square/pos_images \
    --camera agentview_image

In [ ]:
# Adjust --exclude indices after inspecting images above
!python helper_scripts/filter_guidance_dataset.py \
    --input dataset/tight_square/pos_examples.hdf5 \
    --exclude 2 5

The `ood_configs/tight_square.json` config file points to this guidance dataset and reuses the standard initial states. Verify it:

```json
{
    "env_setup": {},
    "use_neg": false,
    "pos_examples": "dataset/tight_square/pos_examples.hdf5",
    "neg_examples": null,
    "reset_poses": "dataset/square_splits/square_initial_states.json"
}
```

## 2.5 DynaGuide — Tight Fit with Visual Guidance

Pass both `--hole_x`/`--hole_y` and `--exp_setup_config` so that `run_dynaguide.py` uses the tight-nut environment and the matching guidance examples:

In [ ]:
!python dyna_scripts/run_dynaguide.py \
    --output_folder eval/tight_square/ \
    --dp_model models/dp_ep500.pth \
    --dynamics_model models/model_epoch_999_augmented_200.pth \
    --camera_names agentview_image \
    --exp_setup_config configs/experiments/ood_configs/tight_square.json \
    --hole_x 0.018 --hole_y 0.018 \
    --n_rollouts 100 --horizon 200 --seed 0 \
    --render_denoising

## 2.6 Sweeping Hole Size

Run the baseline across multiple hole sizes to measure how success rate degrades as clearance decreases. Each call produces a separate stats JSON for easy comparison:

In [ ]:
# Clearance sweep — original nut (no flag), 2 mm, 1 mm clearance per side
!python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --env NutAssemblySquare \
    --n_rollouts 100 --horizon 200 \
    --initial_states_json dataset/square_splits/square_initial_states.json \
    --json_path eval/tight_square/sweep_original_stats.json \
    --seed 0 \
    --video_path eval/tight_square/sweep_original.mp4

In [ ]:
!python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --n_rollouts 100 --horizon 200 \
    --initial_states_json dataset/square_splits/square_initial_states.json \
    --hole_x 0.017 --hole_y 0.017 \
    --json_path eval/tight_square/sweep_017_stats.json \
    --seed 0 \
    --video_path eval/tight_square/sweep_017.mp4

In [ ]:
!python dyna_scripts/run_trained_agent_in_order.py \
    --agent models/dp_ep500.pth \
    --n_rollouts 100 --horizon 200 \
    --initial_states_json dataset/square_splits/square_initial_states.json \
    --hole_x 0.0165 --hole_y 0.0165 \
    --json_path eval/tight_square/sweep_0165_stats.json \
    --seed 0 \
    --video_path eval/tight_square/sweep_0165.mp4

## 2.7 DSRL Tight Square

This guide expects that you have already worked trough *DSRL.ipynb* and therefore are familiar with handeling DSRL in this environment.

After having obtained a suitable noise vector following *DSRL.ipynb* use the following command to evaluute DSRLs performance when being confronted with the tight nut environment:
```bash
python dsrl_scripts/dsrl_test.py \
    --dp_ckpt dataset/YOUR_PREPARED_DP_CKPT.pth \
    --noise_actor_ckpt dataset/YOUR_NOISE_ACTOR.pth \
    --n_episodes 100 \
    --max_steps 200 \
    --video_path ./RESULT_FOLDER \
    --env NutAssemblyTightSquare
```

**CAUTION**</br>
This command directuly uses the peg / hole size defined in `/robosuite/robosuite/environments/manipulation/nut_assembly.py`.</br>
If you want to change this to evaluated different sizes, go directly in this file and change hole_x & hole_y. Keep in mind that the diameter of the whole always is 2*hole_x or 2*hole_y. (With hole_x=0.018 the hole therefore has a diameter of 0.036m == 36mm)
```bash
def __init__(self, hole_x=0.018, hole_y=0.018, **kwargs):
        self._hole_x = float(hole_x)
        self._hole_y = float(hole_y)
```

## 2.8 Expected Results Summary

| Condition | Environment | hole_x/y | Clearance | Expected outcome |
|---|---|---|---|---|
| In-distribution reference | `NutAssemblySquare` | 0.02275 | 6.75 mm | Current best success rate |
| Tight baseline | `NutAssemblyTightSquare` | 0.018 | 2.0 mm | Drop vs. in-distribution |
| Very tight baseline | `NutAssemblyTightSquare` | 0.017 | 1.0 mm | Further drop |
| Tight + DynaGuide | `NutAssemblyTightSquare` | 0.018 | 2.0 mm | Recovery toward in-distribution |

Both conditions (8.3 and 8.5) iterate through the **same 100 initial states** from `square_initial_states.json`. The only variable is whether DynaGuide's visual guidance is active. Compare the `Success_Rate` fields in the output JSON files.

---
# 3. Insertion Pose Steering Experiment

The goal of this experiment is to steer DynaGuide to insert the nut from a specific direction.</br>
For example, we used DynaGuide to steer the diffusion policy to insert the nut preferably from the left, but avoid insertion from the right. 

<img src="pictures/InsertionPose.png" height="200">

The setup of this experiment is rather simple, as it is very close to the standard DynaGuide rollouts.

Using `experiment_automation/compare_guidance_directions.py` you can start a DynaGuide evaluation loop, that does DynaGuide rollouts steering on every direction (LEFT / BACK / RIGHT / ALL) separately, resulting in a total of four rollouts.

You need to provide
- guidance direction datasets for each direction placed in ORIGINAL_DATASETS_DIR
- diffusion policy specified in RUN_PARAMS
- dynamics model specified in RUN_PARAMS
- set N to the desired number of guidance samples that shall be used
- also set the correct CONDA_ENV in run_dynaguide_experiment
- lastly, configure `configs/dynaGuide/dynaguide_env_config.json` to include the correct path to the *reset_poses*. (The path for the pos_examples will be adjusted by the script automatically and therefore does not have to hold a valid value. Negative steering is NOT supported by this script, but can be included manually by setting the parameter TRUE and passing a valid path in the config)




---
# Next steps


After having successfully conducted those experiments, you can compare the achieved results with DSRL to challange your findings.

For this, see [DSRL](DSRL.ipynb).